# Kaggriculture: teacher replays and Behavior Cloning

Run the cells from top to bottom. Before starting, add `KAGGLE_API_TOKEN` in **Colab > Secrets** and allow notebook access to it. A CPU runtime is sufficient for the current streaming SGD model.

In [ ]:
REPOSITORY = "https://github.com/GrigoriiIurev/Kaggriculture.git"
!rm -rf /content/Kaggriculture
!git clone -q {REPOSITORY} /content/Kaggriculture
%cd /content/Kaggriculture
!python3 -m pip install -q -r requirements.txt
!python3 -m pip install -q -U kaggle

## Kaggle authorization
Create an API token on the Kaggle settings page. Store only its value in the Colab secret named `KAGGLE_API_TOKEN`; never paste it into the notebook or GitHub.

In [ ]:
import os
from google.colab import userdata

token = userdata.get("KAGGLE_API_TOKEN")
assert token, "Add KAGGLE_API_TOKEN to Colab Secrets first"
os.environ["KAGGLE_API_TOKEN"] = token
!kaggle competitions list -s kaggriculture

## Choose the training sample
The recommended first run uses the newest 30 games from the best active submission of each of the top 10 players. Increase the numbers only after this run completes successfully.

In [ ]:
import subprocess

TOP_PLAYERS = 10
REPLAYS_PER_PLAYER = 50
BEST_SUBMISSION_ONLY = True

command = [
    "python3", "-u", "download_top_replays.py",
    "--top-players", str(TOP_PLAYERS),
    "--max-replays-per-player", str(REPLAYS_PER_PLAYER),
]
if BEST_SUBMISSION_ONLY:
    command.append("--best-submission-only")

process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)
return_code = process.wait()
if return_code:
    raise subprocess.CalledProcessError(return_code, command)

In [ ]:
!python3 -u build_teacher_dataset.py --winner-only --worker-only
!du -sh data/teacher_replays data/teacher_processed

## Train and evaluate
The model reads compressed worker samples in batches, so it does not load the entire dataset into RAM.

In [ ]:
!mkdir -p /content/kaggriculture_results
!python3 -m src.kaggriculture.learning.train_behavior_cloning \
  --dataset data/teacher_processed/worker_dataset.jsonl.gz \
  --manifest data/teacher_processed/worker_manifest.json \
  --transitions data/teacher_processed/transitions.jsonl.gz \
  --model /content/kaggriculture_results/teacher_worker_bc.npz \
  --report /content/kaggriculture_results/teacher_worker_bc_report.json \
  --policy-report /content/kaggriculture_results/teacher_worker_bc_policy_report.json \
  --epochs 3 --batch-size 1024

## Save results to Google Drive
Only the model and reports are saved by default. Set `SAVE_PROCESSED_DATASET = True` if Drive has enough free space and you also want a compressed archive of the processed teacher dataset.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

SAVE_PROCESSED_DATASET = False
drive.mount("/content/drive")
destination = Path("/content/drive/MyDrive/Kaggriculture/results")
destination.mkdir(parents=True, exist_ok=True)
for path in Path("/content/kaggriculture_results").iterdir():
    shutil.copy2(path, destination / path.name)
if SAVE_PROCESSED_DATASET:
    archive = shutil.make_archive(
        "/content/teacher_processed", "gztar", "data/teacher_processed"
    )
    shutil.copy2(archive, destination / Path(archive).name)
print(f"Saved to {destination}")